# 2장. CrewAI로 RAG 에이전트 구축

## 라이브러리 가져오기

에이전트와 작업을 생성하고 관리하는 데 필요한 라이브러리들을 가져온다.

In [1]:
%pip install -q crewai crewai-tools

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
open-webui 0.6.18 requires chromadb==0.6.3, but you have chromadb 0.5.23 which is incompatible.
open-webui 0.6.18 requires onnxruntime==1.20.1, but you have onnxruntime 1.22.0 which is incompatible.
open-webui 0.6.18 requires posthog==5.4.0, but you have posthog 3.25.0 which is incompatible.
open-webui 0.6.18 requires pypdf==4.3.1, but you have pypdf 5.9.0 which is incompatible.
transformers 4.54.0 requires tokenizers<0.22,>=0.21, but you have tokenizers 0.20.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Crew, Task, Agent, LLM
from crewai_tools import RagTool

/opt/miniconda3/envs/lecture/lib/python3.11/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


## 에이전트의 대규모 언어 모델(LLM) 정의

In [4]:
import os
from dotenv import load_dotenv  

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
llm = LLM(
    model="openai/gpt-4.1-mini", 
    max_tokens=1024
)

## RAG 도구 정의

In [16]:
config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "gpt-4",
        }
    },
    "embedding_model": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-ada-002"
        }
    }
}

In [17]:
rag_tool = RagTool(
    config=config,
    chunk_size=1200,
    chunk_overlap=200
)
rag_tool.add("dataset/first_aid_manual.pdf", data_type="pdf_file")

In [18]:
rag_tool

RagTool(name='Knowledge base', description="Tool Name: Knowledge base\nTool Arguments: {'query': {'description': None, 'type': 'str'}}\nTool Description: A knowledge base that can be used to answer questions.", env_vars=[], args_schema=<class 'abc.RagToolSchema'>, description_updated=False, cache_function=<function BaseTool.<lambda> at 0x114506340>, result_as_answer=False, max_usage_count=None, current_usage_count=0, summarize=False, adapter=EmbedchainAdapter(embedchain_app=<embedchain.app.App object at 0x1067f4b50>, summarize=False), config={'llm': {'provider': 'openai', 'config': {'model': 'gpt-4'}}, 'embedding_model': {'provider': 'openai', 'config': {'model': 'text-embedding-ada-002'}}})

## 응급 에이전트 정의

In [19]:
aid_agent = Agent(
    role="응급처치 가이드",
    goal="학교에서 응급 환자 발생 시 응급처치하는 방법을 가이드한다.",
    backstory="당신은 학교 내에서 응급 환자 발생 시 응급처치를 안내하는 에이전트다",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[rag_tool],
    max_retry_limit=5
)

## 에이전트 작업 정의

In [20]:
task1 = Task(
        description='학교 내에서 응급환자 발생 시 응급처치는 어떻게 할 것인가?',
        expected_output = "사용자의 질문에 대한 구체적인 답변",
        agent=aid_agent
)

## 응급가이드 에이전트 실행

In [21]:
crew = Crew(agents=[aid_agent], tasks=[task1], verbose=True)
task_output = crew.kickoff()
print(task_output)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: edd90c9b-b005-47c8-b0d4-390ec216d949                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 응급처치 가이드                                                                                         │
│                                                                                                                 │
│  Task: 학교 내에서 응급환자 발생 시 응급처치는 어떻게 할 것인가?                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Failed                                                                                              │
│  Name: Knowledge base                                                                                           │
│  Error: Arguments validation failed: 1 validation error for RagToolSchema                                       │
│  query                                                                                                          │
│    Field required [type=missing, input_value={'description': '응급...2381', 'metadata': {}}}},                  │
│  input_type=dict]                                                                                               │
│      For further information visit https://errors.pydantic.dev/2.11/v/missing                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Failed                                                                                              │
│  Name: Knowledge base                                                                                           │
│  Error: Arguments validation failed: 1 validation error for RagToolSchema                                       │
│  query                                                                                                          │
│    Field required [type=missing, input_value={'description': '응급...2381', 'metadata': {}}}},                  │
│  input_type=dict]                                                                                               │
│      For further information visit https://errors.pydantic.dev/2.11/v/missing                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Usage Failed                                                                                              │
│  Name: Knowledge base                                                                                           │
│  Error: Arguments validation failed: 1 validation error for RagToolSchema                                       │
│  query                                                                                                          │
│    Field required [type=missing, input_value={'description': '응급...2381', 'metadata': {}}}},                  │
│  input_type=dict]                                                                                               │
│      For further information visit https://errors.pydantic.dev/2.11/v/missing                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

 Maximum iterations reached. Requesting final answer.


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 응급처치 가이드                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  응급환자 발생 시 학교에서의 응급처치 방법은 다음과 같습니다:                                                   │
│                                                                                                                 │
│  1. **안전 확보:** 사고 현장의 안전을 확인한 후, 부상자에게 접근합니다. 주변에 위험 요소가 없는지 확인하고,     │
│  자신 및 다른 사람의 안전을 우선 고려해야 합니다.                                                               │
│                                                                                                                 │
│  2. **응급 구조 요청:** 주변 사람에게 도움을 요청하거나, 직접 응급 구조 서비스(119)에 연락하여 도움을           │
│  요청합니다. 부상자가 의식이 없다면, 즉시 119나 학교의 응급 연락처에 전화해야 합니다.                           │
│                                                                                                                 │
│  3. **부상자의 상태 확인:** 부상자가 의식이 있는지, 호흡이 정상인지 확인합니다. 부상자가 의식이 없으면 기도를   │
│  확보하고 인공호흡을 실시해야 할 수도 있습니다.                                                                 │
│                                                                                                                 │
│  4. **출혈이 있을 경우:** 출혈이 있는 경우 깨끗한 천이나 드레싱을 사용해 지혈을 시도합니다. 가능한 빨리 출혈    │
│  부위를 압박하여 혈액이 흐르는 것을 막습니다.                                                                   │
│                                                                                                                 │
│  5. **부목 사용:** 골절이나 염좌가 의심될 경우, 해당 부위를 움직이지 않도록 하고, 필요시 부목을 사용해          │
│  고정합니다.                                                                                                    │
│                                                                                                                 │
│  6. **상태 유지:** 부상자의 상태를 주의 깊게 관찰하며, 호흡이나 맥박이 불규칙할 경우, 지속적으로 상태를         │
│  체크합니다.                                                                                                    │
│                                                                                                                 │
│  7. **안정시키기:** 부상자를 편안하게 안정시키고, 필요하다면 담요 등을 덮어 체온을 유지하게 합니다.             │
│                                                                                                                 │
│  8. **의사에게 인계:** 응급 구조대가 도착하면 부상자의 상태와 모든 정보를 상세히 전달하여 치료를 받을 수        │
│  있도록 합니다.                                                                                                 │
│                                                                                                                 │
│  이 절차는 일반적인 가이드라인이며, 상황에 따라 달라질 수 있습니다. 항상 자신의 안전을 최우선으로 하고,         │
│  응급처치 교육을 사전에 받아두는 것이 좋습니다.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 568b4773-e6bd-4d1a-9b81-d72ab2dffb7a                                                                     │
│  Agent: 응급처치 가이드                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: edd90c9b-b005-47c8-b0d4-390ec216d949                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: 응급환자 발생 시 학교에서의 응급처치 방법은 다음과 같습니다:                                     │
│                                                                                                                 │
│  1. **안전 확보:** 사고 현장의 안전을 확인한 후, 부상자에게 접근합니다. 주변에 위험 요소가 없는지 확인하고,     │
│  자신 및 다른 사람의 안전을 우선 고려해야 합니다.                                                               │
│                                                                                                                 │
│  2. **응급 구조 요청:** 주변 사람에게 도움을 요청하거나, 직접 응급 구조 서비스(119)에 연락하여 도움을           │
│  요청합니다. 부상자가 의식이 없다면, 즉시 119나 학교의 응급 연락처에 전화해야 합니다.                           │
│                                                                                                                 │
│  3. **부상자의 상태 확인:** 부상자가 의식이 있는지, 호흡이 정상인지 확인합니다. 부상자가 의식이 없으면 기도를   │
│  확보하고 인공호흡을 실시해야 할 수도 있습니다.                                                                 │
│                                                                                                                 │
│  4. **출혈이 있을 경우:** 출혈이 있는 경우 깨끗한 천이나 드레싱을 사용해 지혈을 시도합니다. 가능한 빨리 출혈    │
│  부위를 압박하여 혈액이 흐르는 것을 막습니다.                                                                   │
│                                                                                                                 │
│  5. **부목 사용:** 골절이나 염좌가 의심될 경우, 해당 부위를 움직이지 않도록 하고, 필요시 부목을 사용해          │
│  고정합니다.                                                                                                    │
│                                                                                                                 │
│  6. **상태 유지:** 부상자의 상태를 주의 깊게 관찰하며, 호흡이나 맥박이 불규칙할 경우, 지속적으로 상태를         │
│  체크합니다.                                                                                                    │
│                                                                                                                 │
│  7. **안정시키기:** 부상자를 편안하게 안정시키고, 필요하다면 담요 등을 덮어 체온을 유지하게 합니다.             │
│                                                                                                                 │
│  8. **의사에게 인계:** 응급 구조대가 도착하면 부상자의 상태와 모든 정보를 상세히 전달하여 치료를 받을 수        │
│  있도록 합니다.                                                                                                 │
│                                                                                                                 │
│  이 절차는 일반적인 가이드라인이며, 상황에 따라 달라질 수 있습니다. 항상 자신의 안전을 최우선으로 하고,         │
│  응급처치 교육을 사전에 받아두는 것이 좋습니다.                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

응급환자 발생 시 학교에서의 응급처치 방법은 다음과 같습니다:

1. **안전 확보:** 사고 현장의 안전을 확인한 후, 부상자에게 접근합니다. 주변에 위험 요소가 없는지 확인하고, 자신 및 다른 사람의 안전을 우선 고려해야 합니다.

2. **응급 구조 요청:** 주변 사람에게 도움을 요청하거나, 직접 응급 구조 서비스(119)에 연락하여 도움을 요청합니다. 부상자가 의식이 없다면, 즉시 119나 학교의 응급 연락처에 전화해야 합니다.

3. **부상자의 상태 확인:** 부상자가 의식이 있는지, 호흡이 정상인지 확인합니다. 부상자가 의식이 없으면 기도를 확보하고 인공호흡을 실시해야 할 수도 있습니다.

4. **출혈이 있을 경우:** 출혈이 있는 경우 깨끗한 천이나 드레싱을 사용해 지혈을 시도합니다. 가능한 빨리 출혈 부위를 압박하여 혈액이 흐르는 것을 막습니다.

5. **부목 사용:** 골절이나 염좌가 의심될 경우, 해당 부위를 움직이지 않도록 하고, 필요시 부목을 사용해 고정합니다.

6. **상태 유지:** 부상자의 상태를 주의 깊게 관찰하며, 호흡이나 맥박이 불규칙할 경우, 지속적으로 상태를 체크합니다.

7. **안정시키기:** 부상자를 편안하게 안정시키고, 필요하다면 담요 등을 덮어 체온을 유지하게 합니다.

8. **의사에게 인계:** 응급 구조대가 도착하면 부상자의 상태와 모든 정보를 상세히 전달하여 치료를 받을 수 있도록 합니다.

이 절차는 일반적인 가이드라인이며, 상황에 따라 달라질 수 있습니다. 항상 자신의 안전을 최우선으로 하고, 응급처치 교육을 사전에 받아두는 것이 좋습니다.
